# MOSAIC GBM: TIL-high and TIL-low stratification

Assigns TIL-high and TIL-low groups from the spot-level TIL+ calls in the TIL zone annotation file and regenerates Supp Fig 5a, Supp Fig 5b and Fig 5c. A Visium section is TIL-high when its TIL+ spot fraction reaches the upper quartile across all sections and TIL-low otherwise, so a section without any TIL+ spot is TIL-low. This is the assignment saved as TIL_groups.csv and read by the ECM notebook. Fig 5c compares TIL zones and leaves out the sections without a TIL+ spot, because border zones are defined around TIL+ spots. Supp Fig 5a and 5b describe whole sections and keep them. Fig 5b recomputes the cutoff on sections with at least 100 spots and is in survival_supplement.

**Produces**: the TIL-high / TIL-low assignment (cut-off 0.030350, 57 TIL-high / 171 TIL-low over
228 sections; 57 / 148 in zone comparisons), Supp Fig 5a, Supp Fig 5b (Spearman ρ 0.600,
Fisher OR 14.57), Fig 5c and the probe-batch table.

**Origin**: Spencer Watson's `TIL_supplement.ipynb` (dropped in the repo root, Nov 2026), all cells
in his order. Changes:
- Setup cell split into one imports cell and one Config cell. His helpers are now in the package and
  imported: `parse_ids` → `gbmcollaboration.paper.ids`; `load_til_zones`, `til_groups`, `zone_groups`
  and `VISIUM_EXCLUSIONS` / `visium_section_excluded` → `gbmcollaboration.paper.til`
  (`til_zones()`, `til_groups()`, `zones_with_groups()`; same rules, exclusions applied through
  `gbmcollaboration.apply_exclusions`); `scanpy_style`, `save` → `gbmcollaboration.paper.output`.
  `til_groups()` returns one row per section indexed by section name, with `TIL_group` and
  `TIL_group_zones` (his `zone_groups`): `GROUPS` is re-indexed by `visium_id` and given his
  `sample` and `TIL_percent` columns so his cells run unchanged. The print of sections removed by
  the loader rules is gone (none are; `til_zones()` applies the rules).
- The module `til` is imported as `til_zones`/`til_groups`/`zones_with_groups` functions, because his
  Supp Fig 5b cells use a variable named `til`.
- Fig 5c cell2location loader (`load_sample`, `load_center`, listing the S3 prefixes) →
  `visium.load_c2l()` over the 228 sections of the zone file (same abundances; his loader also read
  sections outside the zone file, which the reindex dropped). Annotation-key cell →
  `annotation.celltype_key(efs_copy=True)` (the EFS copy he read).
- Outputs: `OUT = output.setup("16_TIL_supplement")` (EFS `figures/paper/16_TIL_supplement/`) instead
  of `revision_output/TIL`; `SAVE_FIGURES` flag dropped. Versions cell → `output.versions`.
- Added: the check of Fig 5c against the published table (`Analysis_Repo/Notebooks/
  TIL_zone_celltype_binary_bubble.csv`). TIL+, BZ1 and BZ2 are expected identical; TIL- differs
  because the published run kept the 23 sections without a TIL+ spot as TIL-low (D3).

In [ ]:
import gc

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib_venn import venn2, venn2_circles
from scipy.stats import fisher_exact, mannwhitneyu, pointbiserialr, spearmanr
from statsmodels.stats.multitest import multipletests

from gbmcollaboration.definitions import PROBE_ISSUE_CHUV, PROBE_ISSUE_UKER
from gbmcollaboration.paper import annotation, config, output, visium
from gbmcollaboration.paper.ids import parse_ids
from gbmcollaboration.paper.output import scanpy_style
from gbmcollaboration.paper.til import til_groups, zones_with_groups

In [ ]:
# Config
OUT = output.setup("16_TIL_supplement")
PATHS = {
    "til_zones": config.TIL_ZONES_CSV,
    "ct": config.CT_ANNOTATIONS_CSV,
    "fig5c_published": config.EFS / "Analysis_Repo/Notebooks/TIL_zone_celltype_binary_bubble.csv",
}
QUANTILE_CUTOFF = 0.75
CENTERS = ["CH", "ER", "CR"]
CENTER_LABELS = {"CH": "CHUV", "ER": "UKER", "CR": "Charité"}


PANELS = {"suppfig5a_til_scatter": "Supp Fig 5a", "suppfig5b_ct0_til_venn": "Supp Fig 5b",
          "fig5c_til_zone_celltype_bubble": "Fig 5c"}   # manuscript ids, also written to OUT_ROOT/panels/


def save(fig, stem, data):
    output.save(OUT, stem, fig, data, panel=PANELS.get(stem))


ZONES_DF = zones_with_groups()   # spots; TIL_group is left out for sections without a TIL+ spot
GROUPS = til_groups(QUANTILE_CUTOFF)
CUTOFF = GROUPS.attrs["cutoff"]
GROUPS = GROUPS.reset_index(names="sample")
GROUPS["TIL_percent"] = 100.0 * GROUPS["TIL_fraction"]
GROUPS["visium_id"] = GROUPS["sample"].str.replace(r"_vis$", "", regex=True)
GROUPS = GROUPS.set_index("visium_id")
PROBE_BLOCKS = set(parse_ids(list(PROBE_ISSUE_CHUV) + list(PROBE_ISSUE_UKER))["block"].dropna())
GROUPS["probe_batch"] = GROUPS["block"].isin(PROBE_BLOCKS)
print(f"{len(ZONES_DF):,} spots, {int(ZONES_DF['TIL_positive'].sum()):,} TIL+ spots, {len(GROUPS)} sections; "
      f"groups {GROUPS['TIL_group'].value_counts().to_dict()}")

## Group assignment

In [ ]:
# TIL-high and TIL-low sections, checked against the published assignment

# Config
PUBLISHED_CUTOFF = 0.030350
PUBLISHED_COUNTS = {"TIL_high": 57, "TIL_low": 171}

print(f"TIL_fraction cutoff for TIL_high (Q={QUANTILE_CUTOFF:.2f}): {CUTOFF:.6f} (published {PUBLISHED_CUTOFF:.6f})")
check = pd.DataFrame({"published": pd.Series(PUBLISHED_COUNTS),
                      "this run": GROUPS["TIL_group"].value_counts()}).fillna(0).astype(int)
check["match"] = check["published"] == check["this run"]
display(check)
if not check["match"].all():
    print("The groups differ from the published assignment. Check the zone file and exclusion lists before continuing.")

rows = []
for c in CENTERS + ["all"]:
    s = GROUPS if c == "all" else GROUPS[GROUPS["center"] == c]
    rows.append({"center": "all" if c == "all" else CENTER_LABELS[c], "sections": len(s),
                 "TIL-high": int((s["TIL_group"] == "TIL_high").sum()),
                 "TIL-low": int((s["TIL_group"] == "TIL_low").sum()),
                 "TIL-low without a TIL+ spot": int(((s["TIL_group"] == "TIL_low") & (s["TIL_spots"] == 0)).sum()),
                 "median TIL+ fraction": round(s["TIL_fraction"].median(), 4)})
display(pd.DataFrame(rows).set_index("center"))
zg = GROUPS["TIL_group_zones"]
print(f"Zone comparisons: {int((zg == 'TIL_high').sum())} TIL-high, {int((zg == 'TIL_low').sum())} TIL-low, "
      f"{int(zg.isna().sum())} sections without a TIL+ spot left out")
# Same columns as the published TIL_groups.csv, so the two files can be compared row by row
GROUPS.reset_index()[["visium_id", "TIL_spots", "Total_Spots", "TIL_fraction", "TIL_percent", "TIL_group"]].to_csv(
    OUT / "TIL_groups.csv", index=False)

## Supp Fig 5a

TIL+ spot count against TIL+ fraction per sample on log axes, with the upper-quartile cutoffs and marginal distributions. Zeros are floored so they stay visible on the log axes.

In [ ]:
# Supp Fig 5a, TIL+ count versus fraction, log-scaled, with quartile cutoffs

# Config
KEY_LEN = 9              # sample key is the Visium sample without '_vis', first 9 characters
QUANTILE = 0.75
USE_LOG = True           # log axes handle the right skew, set False for linear axes
N_BINS = 40
HIST_ALPHA = 0.6
COL_HIGH = "#D55E00"
COL_REST = "#6699CC"

key = ZONES_DF["sample"].str.replace(r"_vis$", "", regex=True).str[:KEY_LEN]
spot = pd.DataFrame({"key": key.values, "til": ZONES_DF["TIL_positive"].astype(float).values}).dropna(subset=["til"])
g = spot.groupby("key").agg(TIL_spots=("til", "sum"), Total_Spots=("til", "size"))
g["TIL_fraction"] = g["TIL_spots"] / g["Total_Spots"]

frac_cut = g["TIL_fraction"].quantile(QUANTILE)
count_cut = g["TIL_spots"].quantile(QUANTILE)
g["TIL_high"] = g["TIL_fraction"] >= frac_cut
hi = g["TIL_high"].values
print(f"Patients: {len(g)}  |  TIL-high (frac >= Q75): {int(hi.sum())}")
print(f"fraction Q75 = {frac_cut:.4f}  |  count Q75 = {count_cut:.0f}  |  "
      f"zero-TIL patients: {int((g['TIL_spots'] == 0).sum())}")

# Zeros are floored so they remain visible on a log axis
if USE_LOG:
    cfloor = 0.5
    pos_frac = g.loc[g["TIL_fraction"] > 0, "TIL_fraction"].min()
    ffloor = pos_frac / 2 if np.isfinite(pos_frac) else 1e-4
    xv = g["TIL_spots"].clip(lower=cfloor).values.astype(float)
    yv = g["TIL_fraction"].clip(lower=ffloor).values.astype(float)
    xcut, ycut = max(count_cut, cfloor), max(frac_cut, ffloor)
    bx = np.logspace(np.log10(xv.min()), np.log10(xv.max()), N_BINS)
    by = np.logspace(np.log10(yv.min()), np.log10(yv.max()), N_BINS)
    xlab, ylab = "TIL+ spots (log scale)", "TIL+ fraction (log scale)"
else:
    xv = g["TIL_spots"].values.astype(float)
    yv = g["TIL_fraction"].values.astype(float)
    xcut, ycut = count_cut, frac_cut
    bx = np.linspace(xv.min(), xv.max(), N_BINS)
    by = np.linspace(yv.min(), yv.max(), N_BINS)
    xlab, ylab = "TIL+ spots", "TIL+ fraction"

fig = plt.figure(figsize=(6.6, 6.6))
gs = GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4], wspace=0.04, hspace=0.04)
ax = fig.add_subplot(gs[1, 0])
axt = fig.add_subplot(gs[0, 0], sharex=ax)
axr = fig.add_subplot(gs[1, 1], sharey=ax)
if USE_LOG:
    ax.set_xscale("log")
    ax.set_yscale("log")

ax.scatter(xv[~hi], yv[~hi], s=24, c=COL_REST, edgecolor="white", linewidth=0.4, label="TIL-low (rest)", zorder=3)
ax.scatter(xv[hi], yv[hi], s=28, c=COL_HIGH, edgecolor="white", linewidth=0.4, label="TIL-high (top quartile)",
           zorder=4)
ax.axvline(xcut, color="#333", ls="--", lw=1.0, zorder=2)
ax.axhline(ycut, color="#333", ls="--", lw=1.0, zorder=2)
ax.set_xlabel(xlab, fontsize=10)
ax.set_ylabel(ylab, fontsize=10)
ax.legend(fontsize=8, loc="lower right", framealpha=0.9)
for sp in ["top", "right"]:
    ax.spines[sp].set_visible(False)

axt.hist([xv[~hi], xv[hi]], bins=bx, stacked=True, color=[COL_REST, COL_HIGH], alpha=HIST_ALPHA, edgecolor="white",
         linewidth=0.3)
axt.axvline(xcut, color="#333", ls="--", lw=1.0)
axt.set_ylabel("patients", fontsize=8)
axt.tick_params(labelbottom=False, labelsize=7)
for sp in ["top", "right", "left"]:
    axt.spines[sp].set_visible(False)

axr.hist([yv[~hi], yv[hi]], bins=by, stacked=True, orientation="horizontal", color=[COL_REST, COL_HIGH],
         alpha=HIST_ALPHA, edgecolor="white", linewidth=0.3)
axr.axhline(ycut, color="#333", ls="--", lw=1.0)
axr.set_xlabel("patients", fontsize=8)
axr.tick_params(labelleft=False, labelsize=7)
for sp in ["top", "right", "bottom"]:
    axr.spines[sp].set_visible(False)

fig.suptitle("TIL-high defined as top quartile of TIL+ fraction", fontsize=11, y=0.98)
ax.annotate(f"fraction Q75 = {frac_cut:.3f}\ncount Q75 = {count_cut:.0f}\nTIL-high: {int(hi.sum())}/{len(g)}",
            xy=(0.03, 0.97), xycoords="axes fraction", ha="left", va="top", fontsize=8, color="#333")
save(fig, "suppfig5a_til_scatter", g.reset_index())
plt.show()
plt.close(fig)

## Supp Fig 5b

Whether high-CT_0 samples coincide with TIL-high samples, over every ST sample with both a cell type neighborhood annotation and a TIL group. CT_0 fraction is compared with TIL+ fraction by Spearman correlation and with TIL group by point-biserial correlation, and CT_0 split at its median is crossed with TIL group in Fisher's exact test.

In [ ]:
# Do high-CT_0 patients coincide with TIL-high patients? Continuous and categorical, full cohort

# Config
NH = 0

# Per-sample CT_0 fraction, keyed like TIL_groups.csv (sample without '_vis')
ann = pd.read_csv(PATHS["ct"], usecols=["sample", "celltype_neighborhood"])
ann["key"] = ann["sample"].str.replace(r"_vis$", "", regex=True)
ann["is_nh"] = (ann["celltype_neighborhood"] == NH).astype(int)
n0 = ann.groupby("key")["is_nh"].mean().rename("n0_frac")

til = GROUPS.reset_index()[["visium_id", "TIL_group", "TIL_fraction"]]
df = n0.reset_index().merge(til.rename(columns={"visium_id": "key"}), on="key", how="inner")
print(f"Patients matched: {len(df)}  (N0 keys {n0.index.nunique()}, TIL keys {til['visium_id'].nunique()})")

# Continuous, how tightly the two infiltration measures track
rho, p_rho = spearmanr(df["n0_frac"], df["TIL_fraction"])
print(f"\nContinuous (Spearman, N0 fraction vs TIL+ fraction): r={rho:.3f}, p={p_rho:.2e}")

th = df["TIL_group"] == "TIL_high"
r_pb, p_pb = pointbiserialr(th.astype(int), df["n0_frac"])
print(f"Point-biserial (TIL-high vs N0 fraction):            r={r_pb:.3f}, p={p_pb:.2e}")

# Categorical, N0 split at its median as in the survival analysis
hi = df["n0_frac"] >= df["n0_frac"].median()
ct = pd.crosstab(hi, th)
ct.index = ["N0-low", "N0-high"]
ct.columns = ["TIL-low", "TIL-high"]
print("\nContingency (N0 median split x TIL group):")
print(ct.to_string())

orr, p_f = fisher_exact(ct.values)
both, n_hi, n_th = int((hi & th).sum()), int(hi.sum()), int(th.sum())
print(f"\nFisher exact: OR={orr:.2f}, p={p_f:.2e}")
print(f"Overlap: {both} patients are both high-N0 and TIL-high")
print(f"  of TIL-high, {both}/{n_th} = {both / n_th:.0%} are high-N0")
print(f"  of high-N0,  {both}/{n_hi} = {both / n_hi:.0%} are TIL-high")
df.to_csv(OUT / "suppfig5b_ct0_til.csv", index=False)

In [ ]:
# Supp Fig 5b, Venn diagram of high-CT_0 and TIL-high patients

# Config
NH = 0
SET_COLORS = ("#5975A4", "#E07B54")
FONT_RC = {"font.family": "Arial", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "text.color": "black",
           "axes.labelcolor": "black", "xtick.color": "black", "ytick.color": "black"}

# Per-patient CT_0 fraction
ann = pd.read_csv(PATHS["ct"], usecols=["sample", "celltype_neighborhood"])
ann["key"] = ann["sample"].str.replace(r"_vis$", "", regex=True)
ann["is_nh"] = (ann["celltype_neighborhood"] == NH).astype(int)
n0 = ann.groupby("key")["is_nh"].mean().rename("ct0_frac")

til = GROUPS.reset_index()[["visium_id", "TIL_group", "TIL_fraction"]]
df = n0.reset_index().merge(til.rename(columns={"visium_id": "key"}), on="key", how="inner")

hi_ct0 = set(df.loc[df["ct0_frac"] >= df["ct0_frac"].median(), "key"])
hi_til = set(df.loc[df["TIL_group"] == "TIL_high", "key"])

rho, p_rho = spearmanr(df["ct0_frac"], df["TIL_fraction"])
th = (df["TIL_group"] == "TIL_high").astype(int)
r_pb, p_pb = pointbiserialr(th, df["ct0_frac"])

hi = df["ct0_frac"] >= df["ct0_frac"].median()
ct = pd.crosstab(hi, df["TIL_group"] == "TIL_high")
orr, p_f = fisher_exact(ct.values)

only_ct0 = len(hi_ct0 - hi_til)
only_til = len(hi_til - hi_ct0)
both = len(hi_ct0 & hi_til)

with plt.rc_context(FONT_RC):
    fig, ax = plt.subplots(figsize=(5.5, 5))
    v = venn2(subsets=(only_ct0, only_til, both), set_labels=("", ""), set_colors=SET_COLORS, alpha=0.4, ax=ax)
    venn2_circles(subsets=(only_ct0, only_til, both), linewidth=0.8, color="black", ax=ax)
    for text in v.subset_labels:
        if text:
            text.set_fontsize(12)
            text.set_color("black")
    # Set labels sit well outside the circles
    ax.text(-0.62, 0.50, "High-CT_0", fontsize=11, fontweight="semibold", ha="center", color="black")
    ax.text(0.62, 0.50, "TIL-High", fontsize=11, fontweight="semibold", ha="center", color="black")
    ax.set_title("Overlap of High-CT_0 and TIL-High Patients", fontsize=12, fontweight="bold", pad=14, color="black")
    stats_text = "\n".join([
        f"Spearman ρ (CT_0 frac. vs TIL frac.):  {rho:.3f}  (p = {p_rho:.2e})",
        f"Point-biserial r (TIL-high vs CT_0 frac.):  {r_pb:.3f}  (p = {p_pb:.2e})",
        f"Fisher exact OR (2×2 contingency):  {orr:.2f}  (p = {p_f:.2e})",
    ])
    ax.text(0.5, -0.08, stats_text, transform=ax.transAxes, fontsize=8.5, ha="center", va="top", color="black",
            linespacing=1.6)
    total = len(df)
    ax.text(0.5, -0.22, f"n = {total} patients  |  High-CT_0: {len(hi_ct0)}  |  TIL-High: {len(hi_til)}  |  "
            f"Both: {both}", transform=ax.transAxes, fontsize=8.5, ha="center", va="top", color="black")
    ax.set_facecolor("white")
    fig.patch.set_facecolor("white")
    plt.tight_layout()
    save(fig, "suppfig5b_ct0_til_venn", df)
    plt.show()
    plt.close(fig)

## Fig 5c: cell types by TIL zone

Cell2Location absolute abundances are read for every Visium section and aligned to the spots of the TIL zone file. Level 4 abundances are summed to the figure-level cell types of the annotation key and set to zero where they fall below the abundance at which a cell type has a 20% Poisson probability of being present. Each section's spots are averaged within each TIL zone, and TIL-high and TIL-low sections are compared, leaving out sections without a TIL+ spot, per zone and cell type with a two-sided Mann-Whitney U test and Benjamini-Hochberg correction within each zone. The plot shows cell types significant in at least one zone, up to ten per direction.

In [ ]:
# Cell2Location absolute abundances for every Visium section, aligned to the TIL zone spots

# Config
COL_PREFIX = visium.C2L_PREFIX   # "meanscell_abundance_w_sf_"

df_all = visium.load_c2l(sorted(ZONES_DF["sample"].unique()))
print(f"All centers: {len(df_all):,} spots, {df_all.index.get_level_values('sample').nunique()} samples")

# Align on (sample, barcode) of every spot in the zone file, which carries the obs names of the loaded cohort
barcodes = pd.Index(ZONES_DF.index.astype(str)).str.split("_").str[-1]
keys = pd.MultiIndex.from_arrays([ZONES_DF["sample"].values, barcodes.values], names=["sample", "barcode"])
c2l_cols = [c for c in df_all.columns if c.startswith(COL_PREFIX)]
C2L = df_all[c2l_cols].reindex(keys)
C2L.index = ZONES_DF.index
print(f"Aligned: {len(C2L):,} spots x {len(c2l_cols)} cell types; spots without abundances "
      f"{int(C2L.isna().all(axis=1).sum()):,}")
del df_all
_ = gc.collect()

In [ ]:
# Cell type annotation key, level 4 names to figure-level names

CELLTYPE_KEY = annotation.celltype_key(efs_copy=True)

lvl4_in_key = pd.Index(CELLTYPE_KEY.index)
c2l_lvl4 = pd.Index([c.replace(COL_PREFIX, "") for c in C2L.columns])
print(f"Total level4 types in key: {len(lvl4_in_key)}")
print(f"Matching C2L level4 types: {len(c2l_lvl4.intersection(lvl4_in_key))}")
print(f"Level4 types in key not seen in C2L columns: {len(lvl4_in_key.difference(c2l_lvl4))}")

In [ ]:
# Fig 5c, cell type differences between TIL-high and TIL-low sections across TIL zones

# Config
ZONES = ["TIL-", "BZ2", "BZ1", "TIL+"]
FIGURE_COL = "celltype_level4_curated"
PROB_PRESENT = 0.2                     # Poisson presence threshold
APPLY_PRESENCE_THRESHOLD = True
FDR_THRESH = 0.05
TOP_N = 10
SORT_ZONE = "TIL+"
EXCLUDE_CELLTYPES = ["Malignant_other_stressed", "Malignant_other_1", "Malignant_other_2"]
figsize = (18, 4.5)
min_dot_size = 20
max_dot_size = 250
cmap = "RdBu_r"
vmin = None
vmax = None
sig_edgecolor = "black"
sig_edgewidth = 1.5
nonsig_edgecolor = "0.7"
nonsig_edgewidth = 0.5
bg_color = "white"
title_fontsize = 12
axis_label_fontsize = 10
tick_fontsize = 9
celltype_fontsize = 9
celltype_rotation = 45
legend_fontsize = 12

lambda_thresh = -np.log(1.0 - PROB_PRESENT)
obs = ZONES_DF[["sample", "TIL_zone", "TIL_group"]].join(C2L)
obs["Sample_TIL_Group"] = obs["TIL_group"].map({"TIL_high": "TIL-High", "TIL_low": "TIL-Low"})

c2l_cols = [c for c in obs.columns if c.startswith(COL_PREFIX)]
lvl4_names = pd.Index([c.replace(COL_PREFIX, "") for c in c2l_cols])
fig_map = CELLTYPE_KEY[FIGURE_COL].astype(str).reindex(lvl4_names).dropna()
fig_to_cols = {}
for lvl4, fig_label in fig_map.items():
    colname = f"{COL_PREFIX}{lvl4}"
    if colname in obs.columns:
        fig_to_cols.setdefault(fig_label, []).append(colname)

if EXCLUDE_CELLTYPES:
    n_before = len(fig_to_cols)
    fig_to_cols = {k: v for k, v in fig_to_cols.items() if k not in EXCLUDE_CELLTYPES}
    print(f"Excluded {n_before - len(fig_to_cols)} cell types: {EXCLUDE_CELLTYPES}")
    print(f"Remaining: {len(fig_to_cols)} figure-level cell types")

# Test each zone and cell type on per-section means
all_results = []
for zone in ZONES:
    zone_df = obs.loc[obs["TIL_zone"].astype(str) == zone].copy()
    zone_df = zone_df.dropna(subset=["Sample_TIL_Group"])
    if zone_df.empty:
        print(f"WARNING: no spots in zone {zone}")
        continue

    feat = pd.DataFrame(index=zone_df.index)
    for fig_label, cols in fig_to_cols.items():
        present_cols = [c for c in cols if c in zone_df.columns]
        if present_cols:
            vals = zone_df[present_cols].sum(axis=1).to_numpy()
            feat[fig_label] = np.where(np.isfinite(vals), vals, 0.0)
    if APPLY_PRESENCE_THRESHOLD:
        feat = feat.where(feat >= lambda_thresh, 0.0)
    feat = feat.loc[:, (feat != 0).any(axis=0)]
    fig_labels = feat.columns.tolist()

    zone_feat = pd.concat([zone_df[["sample", "Sample_TIL_Group"]], feat], axis=1)
    grouped = zone_feat.groupby("sample", observed=True).agg(
        {**{col: "mean" for col in fig_labels}, "Sample_TIL_Group": "first"}).reset_index()
    g_high = grouped[grouped["Sample_TIL_Group"] == "TIL-High"]
    g_low = grouped[grouped["Sample_TIL_Group"] == "TIL-Low"]
    if len(g_high) < 3 or len(g_low) < 3:
        print(f"WARNING: too few samples in zone {zone} (High={len(g_high)}, Low={len(g_low)})")
        continue

    for col in fig_labels:
        a = g_high[col].dropna().to_numpy()
        b = g_low[col].dropna().to_numpy()
        if a.size < 3 or b.size < 3:
            continue
        _, p = mannwhitneyu(a, b, alternative="two-sided")
        all_results.append({"zone": zone, "cell_type": col, "mean_high": float(a.mean()),
                            "mean_low": float(b.mean()), "diff": float(a.mean()) - float(b.mean()), "pval": p,
                            "n_high": len(a), "n_low": len(b)})

df_res = pd.DataFrame(all_results)

# FDR correction within each zone
df_res["fdr"] = np.nan
for zone in ZONES:
    mask = df_res["zone"] == zone
    if mask.sum() > 0:
        _, fdr, _, _ = multipletests(df_res.loc[mask, "pval"].fillna(1.0), method="fdr_bh")
        df_res.loc[mask, "fdr"] = fdr
df_res["neglog10fdr"] = -np.log10(df_res["fdr"].clip(lower=1e-300))
df_res["significant"] = df_res["fdr"] < FDR_THRESH
print(f"\nTotal tests: {len(df_res)}")
print(f"Significant (FDR<{FDR_THRESH}) per zone:")
print(df_res.groupby("zone")["significant"].sum().to_string())

# Cell types significant in any zone, top N per direction by the difference in their most significant zone
sig_any = df_res[df_res["significant"]]["cell_type"].unique()
print(f"\nCell types significant in >= 1 zone: {len(sig_any)}")
sig_df = df_res[(df_res["cell_type"].isin(sig_any)) & (df_res["significant"])].copy()
best_per_ct = sig_df.loc[sig_df.groupby("cell_type")["fdr"].idxmin()]
pos_all = best_per_ct[best_per_ct["diff"] > 0].sort_values("diff", ascending=False)
neg_all = best_per_ct[best_per_ct["diff"] < 0].sort_values("diff", ascending=True)
pos_cts = pos_all["cell_type"].head(TOP_N).tolist()
neg_cts = neg_all["cell_type"].head(TOP_N).tolist()
print(f"Enriched in TIL-High: {len(pos_all)} total, showing {len(pos_cts)}")
print(f"Enriched in TIL-Low: {len(neg_all)} total, showing {len(neg_cts)}")

sort_ref = df_res[df_res["zone"] == SORT_ZONE].set_index("cell_type")["diff"]
pos_order = sorted(pos_cts, key=lambda ct: sort_ref.get(ct, 0), reverse=True)
neg_order = sorted(neg_cts, key=lambda ct: sort_ref.get(ct, 0), reverse=True)
sort_order = pos_order + neg_order
df_plot = df_res[df_res["cell_type"].isin(sort_order)].copy()
print(f"\nFinal bubble plot: {len(sort_order)} cell types x {len(ZONES)} zones")

if vmin is None or vmax is None:
    abs_max = df_plot["diff"].abs().quantile(0.95)
    vmin, vmax = -abs_max, abs_max
    print(f"Auto color range: [{vmin:.3f}, {vmax:.3f}]")

with scanpy_style(200):
    zone_y = {z: i for i, z in enumerate(ZONES)}
    ct_x = {ct: i for i, ct in enumerate(sort_order)}
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor(bg_color)

    size_vals = df_plot["neglog10fdr"].values
    size_min = np.nanmin(size_vals)
    size_max = np.nanmax(size_vals)
    if size_max == size_min:
        size_max = size_min + 1


    def scale_size(v):
        frac = (v - size_min) / (size_max - size_min)
        return min_dot_size + frac * (max_dot_size - min_dot_size)


    for _, row in df_plot.iterrows():
        x = ct_x.get(row["cell_type"])
        y = zone_y.get(row["zone"])
        if x is None or y is None:
            continue
        ec = sig_edgecolor if row["significant"] else nonsig_edgecolor
        ew = sig_edgewidth if row["significant"] else nonsig_edgewidth
        ax.scatter(x, y, s=scale_size(row["neglog10fdr"]), c=[row["diff"]], cmap=cmap, vmin=vmin, vmax=vmax,
                   edgecolors=ec, linewidths=ew, zorder=3)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.8, aspect=20, pad=0.02)
    cbar.set_label("Mean difference (TIL-High − TIL-Low)", fontsize=axis_label_fontsize)
    cbar.ax.tick_params(labelsize=tick_fontsize)

    ax.set_xticks(range(len(sort_order)))
    ax.set_xticklabels(sort_order, rotation=celltype_rotation, fontsize=celltype_fontsize, ha="right",
                       rotation_mode="anchor")
    ax.set_yticks(range(len(ZONES)))
    ax.set_yticklabels(ZONES, fontsize=tick_fontsize)
    ax.set_xlim(-0.5, len(sort_order) - 0.5)
    ax.set_ylim(-0.5, len(ZONES) - 0.5)
    for i in range(len(sort_order)):
        ax.axvline(i, color="0.94", linewidth=0.3, zorder=1)
    for i in range(len(ZONES)):
        ax.axhline(i, color="0.94", linewidth=0.3, zorder=1)
    ax.set_xlabel("Cell type", fontsize=axis_label_fontsize)
    ax.set_ylabel("TIL zone", fontsize=axis_label_fontsize)
    ax.set_title(f"Cell type differences: TIL-High vs TIL-Low across zones "
                 f"(Mann-Whitney, FDR < {FDR_THRESH} in ≥1 zone, top {TOP_N}/side)", fontsize=title_fontsize)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    legend_vals = [round(v, 1) for v in np.linspace(size_min, size_max, 4)]
    size_handles = [Line2D([0], [0], marker="o", linestyle="None", color="0.5", markersize=np.sqrt(scale_size(v)),
                           label=f"{v:.1f}", markeredgecolor="0.5", markeredgewidth=0.5) for v in legend_vals]
    sig_handle = Line2D([0], [0], marker="o", linestyle="None", color="0.7", markersize=8, markeredgecolor="black",
                        markeredgewidth=1.5, label=f"FDR < {FDR_THRESH}")
    ax.legend(handles=size_handles + [sig_handle], title="-log₁₀(FDR)", title_fontsize=legend_fontsize,
              fontsize=legend_fontsize, loc="upper left", bbox_to_anchor=(1.12, 1.0), frameon=False,
              handletextpad=1.5, labelspacing=1)
    plt.tight_layout()
    save(fig, "fig5c_til_zone_celltype_bubble", df_plot)
    plt.show()
    plt.close(fig)

pivot_diff = df_plot.pivot(index="zone", columns="cell_type", values="diff")[sort_order].round(4)
pivot_fdr = df_plot.pivot(index="zone", columns="cell_type", values="fdr")[sort_order].round(4)
print("\nMean difference (High - Low):")
print(pivot_diff.to_string())
print("\nFDR:")
print(pivot_fdr.to_string())
df_res.to_csv(OUT / "fig5c_all_tests.csv", index=False)

In [ ]:
# Fig 5c compared with the published table
pub = pd.read_csv(PATHS["fig5c_published"])
m = df_res.merge(pub, on=["zone", "cell_type"], how="outer", suffixes=("", "_pub"), indicator=True)
print(f"Rows: {m['_merge'].value_counts().to_dict()}")
for zone, gz in m[m["_merge"] == "both"].groupby("zone"):
    print(f"  {zone}: n_low {sorted(gz['n_low'].unique())} (published {sorted(gz['n_low_pub'].unique())}), "
          f"max |p diff| {np.abs(gz['pval'] - gz['pval_pub']).max():.2g}, "
          f"significance differs for {gz.loc[gz['significant'] != gz['significant_pub'], 'cell_type'].tolist()}")
print(f"Same cell types plotted as published: {set(pub['cell_type']) <= set(df_res['cell_type'])}, "
      f"{len(sort_order)} here")

## Probe batch

gbmcollaboration.definitions flags CHUV and UKER blocks processed with a probe batch noted at quality control. Sections are tabulated by center and batch to check whether the batch tracks TIL-high status within a center. The Fig 5b survival analysis without these blocks is in survival_supplement.

In [ ]:
# TIL-high frequency by center and probe batch

sec = GROUPS.copy()
sec["high"] = (sec["TIL_group"] == "TIL_high").astype(int)
rows = []
for c in CENTERS:
    s = sec[sec["center"] == c]
    tab = pd.crosstab(s["probe_batch"], s["high"]).reindex(index=[False, True], columns=[0, 1], fill_value=0)
    p = fisher_exact(tab.values)[1] if (tab.sum(axis=1) > 0).all() else np.nan
    for flag in (True, False):
        b = s[s["probe_batch"] == flag]
        rows.append({"center": CENTER_LABELS[c], "probe batch": flag, "sections": len(b),
                     "TIL-high": int(b["high"].sum()),
                     "TIL-high %": round(100 * b["high"].mean(), 1) if len(b) else np.nan,
                     "median TIL+ fraction": round(b["TIL_fraction"].median(), 4) if len(b) else np.nan,
                     "Fisher p within center": f"{p:.3g}" if pd.notna(p) else "n/a"})
display(pd.DataFrame(rows).set_index(["center", "probe batch"]))

In [ ]:
# Software versions
print(output.versions(["numpy", "pandas", "scipy", "statsmodels", "matplotlib", "matplotlib-venn", "anndata",
                       "scanpy", "s3fs", "joblib"]).to_string(index=False))